In [4]:
import json
import time
import base64
import os
import re
import subprocess
import pandas as pd
import numpy as np
from io import BytesIO
from pathlib import Path
from PIL import Image
from openai import OpenAI

# =====================================================================
# --- CONFIGURATION VARIABLES ---
# =====================================================================

# How many tasks do you want to run? (Set to None for the whole dataset)
MAX_TASKS_TO_RUN = 1

# Paths
TURTLEBENCH_PATH = Path("/Users/smeh/Desktop/Thesis/DTurtleBench") # Folder containing dataset.jsonl and Tasks/
RESULTS_DIR = Path("TurtleBenchResults")

# Model Settings
MODELS_TO_TEST = ["gemma4:e2b"]
MAX_RETRIES = 3

# API Settings
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    timeout=180.0
)

TURTLEBENCH_SYSTEM_PROMPT = """
INSTRUCTIONS:
1. Write the Python Turtle graphics code to generate the requested image.
2. Output ONLY the raw Python code with no setup for execution.
3. DO NOT wrap the code in markdown formatting (e.g., do not use ```python).
4. DO NOT output any conversational text, explanations, or greetings.
5. Assume the turtle object is named `t` and the math module is imported if needed.
"""

# =====================================================================
# --- PHASE 1: GENERATION FUNCTIONS ---
# =====================================================================

def encode_image_file_to_base64(image_path):
    """Open an image file from disk, convert to JPEG base64 with prefix."""
    try:
        with Image.open(image_path) as img:
            if img.mode != "RGB":
                img = img.convert("RGB")
            buf = BytesIO()
            img.save(buf, format="JPEG", quality=85)
            b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
            return f"data:image/jpeg;base64,{b64}"
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None

def query_openai_client(oai_client, model_name, prompt, base64_image_url):
    """Send an image + prompt via an OpenAI-compatible client."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": base64_image_url}},
        ],
    }]

    start = time.time()
    for attempt in range(MAX_RETRIES):
        try:
            resp = oai_client.chat.completions.create(
                model=model_name, messages=messages,
                stream=False, temperature=0.1,
            )
            return resp.choices[0].message.content.strip(), time.time() - start
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{MAX_RETRIES}: {e}")
            time.sleep(5 * (attempt + 1))
    return "ERROR_MAX_RETRIES_REACHED", time.time() - start

def load_progress(output_csv):
    """Load already-processed task IDs."""
    if output_csv.is_file():
        df = pd.read_csv(output_csv)
        if "task_id" in df.columns:
            return set(df["task_id"].astype(str)), True
    return set(), False

def load_turtlebench_json(dataset_path):
    """Parse the official jsonl manifest and build the task list with GT code."""
    jsonl_path = Path(dataset_path) / "dataset.jsonl"
    tasks_dir = Path(dataset_path) / "Tasks"
    
    if not jsonl_path.exists():
        print(f"❌ Cannot find {jsonl_path}")
        return pd.DataFrame()

    rows = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            task_id = str(data["id"])
            task_folder = tasks_dir / task_id
            
            image_path = task_folder / "image" / f"{task_id}.png"
            query_path = task_folder / "QA" / "text" / "q1.txt"
            code_path = task_folder / "QA" / "code" / "q1_code.txt"
            vars_path = task_folder / "variables.txt"
            
            # Default values
            question_id = "q1"
            query_text = "Create the exact same shape."
            ground_truth_code = "No code found."

            if query_path.exists():
                query_text = query_path.read_text(encoding="utf-8").strip()
                
            if code_path.exists():
                ground_truth_code = code_path.read_text(encoding="utf-8").strip()
                
            # Append critical geometry variables
            if vars_path.exists():
                vars_text = vars_path.read_text(encoding="utf-8").strip()
                query_text += f"\n\nConstraint - Use these variables strictly:\n{vars_text}"

            rows.append({
                "task_id": task_id,
                "question_id": question_id,
                "image_path": str(image_path),
                "query_text": query_text,
                "ground_truth_code": ground_truth_code
            })
            
    return pd.DataFrame(rows)

def run_turtlebench_generation(df, model_name, output_csv, limit=None):
    """Run code generation and save to CSV."""
    print(f"\n🚀 TurtleBench Generation — {model_name}...")
    processed, file_exists = load_progress(output_csv)

    eval_df = df.head(limit) if limit else df
    for _, row in eval_df.iterrows():
        if row['task_id'] in processed:
            continue

        print(f"\nTask {row['task_id']} | Q: {row['question_id']}...")

        b64_url = encode_image_file_to_base64(row["image_path"])
        if not b64_url:
            continue

        prompt = f"{row['query_text']}\n{TURTLEBENCH_SYSTEM_PROMPT}"
        model_response, gen_time = query_openai_client(client, model_name, prompt, b64_url)
        print(f"   ⏱️ Generated in {gen_time:.2f}s")

        # Dictionary exactly matches requested columns
        result_dict = {
            "task_id": row["task_id"], 
            "question_id": row["question_id"],
            "image_path": row["image_path"],
            "query_text": row["query_text"],
            "model_response": model_response,
            "ground_truth_code": row["ground_truth_code"],
            "code_generation_time": round(gen_time, 2)
        }
        
        pd.DataFrame([result_dict]).to_csv(
            output_csv, mode="a", header=not file_exists, index=False
        )
        file_exists = True
        processed.add(row['task_id'])
        time.sleep(2)


# =====================================================================
# --- PHASE 2: EVALUATION FUNCTIONS (IoU + SUBPROCESS) ---
# =====================================================================

def is_code_safe(code_string):
    """Scans the code for dangerous Python commands before execution."""
    forbidden_keywords = [
        'os', 'sys', 'subprocess', 'open', 'eval', 'exec', '__import__', 'shutil'
    ]
    for keyword in forbidden_keywords:
        if re.search(fr'\b{keyword}\b', str(code_string)):
            return False
    return True

def clean_llm_code(code_string):
    """Aggressively removes markdown formatting and conversational text from LLM output."""
    code_string = str(code_string).strip()
    
    # Extract code specifically from markdown blocks if they exist
    match = re.search(r'```(?:python)?(.*?)```', code_string, re.DOTALL | re.IGNORECASE)
    if match:
        code_string = match.group(1).strip()
        
    # Fallback cleanup for lingering backticks
    code_string = code_string.replace('```python', '').replace('```', '')
    return code_string.strip()

def execute_turtle_code(code_string, output_filename):
    """Executes Python Turtle code safely using a subprocess and timeout."""
    # 1. Clean the code first
    cleaned_code = clean_llm_code(code_string)
    
    if not is_code_safe(cleaned_code):
        print("   ⚠️ Execution Blocked: Detected forbidden keywords.")
        return False

    temp_script = f"temp_run_{output_filename.replace('.eps', '')}.py"
    
    script_content = f"""import turtle
import math

# --- Setup ---
try:
    turtle.clearscreen()
except turtle.Terminator:
    pass # Handle edge case where turtle wasn't fully closed previously
    
t = turtle.Turtle()
t.speed(0)
turtle.tracer(0, 0)

# --- LLM Generated Code ---
{cleaned_code}

# --- Teardown & Save ---
turtle.update()
canvas = turtle.getcanvas()
canvas.postscript(file="{output_filename}")

try:
    turtle.bye()
except:
    pass
"""
    
    with open(temp_script, "w", encoding="utf-8") as f:
        f.write(script_content)
        
    try:
        # Run the script
        result = subprocess.run(
            ["python", temp_script], 
            timeout=5, 
            capture_output=True, 
            text=True
        )
        
        # If the script failed, print the exact Python error so we can debug
        if result.returncode != 0:
            error_msg = result.stderr.strip()
            # Just print the last line of the traceback for cleaner logs
            short_err = error_msg.split('\n')[-1] if error_msg else "Unknown Python Crash"
            print(f"   ⚠️ Script Error: {short_err}")
            
        return os.path.exists(output_filename)
        
    except subprocess.TimeoutExpired:
        print("   ⚠️ Execution Failed: Time out (Infinite loop detected).")
        return False
    except Exception as e:
        print(f"   ⚠️ Subprocess Error: {e}")
        return False
    finally:
        if os.path.exists(temp_script):
            os.remove(temp_script)


def calculate_iou(img_path_1, img_path_2):
    """Calculates the Intersection over Union (IoU) of two images."""
    try:
        img1 = Image.open(img_path_1).convert('L')
        img2 = Image.open(img_path_2).convert('L')
        
        img2 = img2.resize(img1.size)
        
        arr1 = np.array(img1)
        arr2 = np.array(img2)
        
        mask1 = arr1 < 200 
        mask2 = arr2 < 200
        
        intersection = np.logical_and(mask1, mask2).sum()
        union = np.logical_or(mask1, mask2).sum()
        
        if union == 0:
            return 0.0 
            
        return intersection / union
    except Exception as e:
        print(f"   ⚠️ IoU Calculation Error: {e}")
        return 0.0

def evaluate_generations(csv_path, dataset_path):
    """Loops through generated code, runs it, and grades it."""
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"❌ Could not read {csv_path}: {e}")
        return

    tasks_dir = Path(dataset_path) / "Tasks"
    ious = []
    eval_percentages = []
    is_correct_list = []
    
    print(f"\n🚀 Starting IoU execution on {csv_path.name}...")
    
    for index, row in df.iterrows():
        task_id = str(row['task_id'])
        code = str(row['model_response'])
        print(f"Evaluating Task {task_id}...")
        
        if "def " not in code and "t." not in code:
            ious.append(0.0)
            eval_percentages.append("0.00%")
            is_correct_list.append(False)
            print("   ❌ No valid code found.")
            continue
            
        gt_image_path = tasks_dir / task_id / "image" / f"{task_id}.png"
        temp_eps = f"temp_{task_id}.eps"
        
        success = execute_turtle_code(code, temp_eps)
        
        if success and os.path.exists(temp_eps):
            iou_score = calculate_iou(gt_image_path, temp_eps)
            is_correct = iou_score >= 0.95 
            
            ious.append(iou_score)
            eval_percentages.append(f"{iou_score * 100:.2f}%")
            is_correct_list.append(is_correct)
            
            print(f"   {'✅' if is_correct else '❌'} IoU Match: {iou_score * 100:.2f}%")
            
            os.remove(temp_eps)
        else:
            ious.append(0.0)
            eval_percentages.append("0.00%")
            is_correct_list.append(False)
            print("   ❌ Execution Failed or produced no visual output.")
            
        time.sleep(1)
        
    df['IoU_Score'] = ious
    df['evaluation_result'] = eval_percentages  # Appends the formatted percentage string
    
    # Reorder columns to ensure they perfectly match the requested layout
    columns_order = [
        "task_id", "question_id", "image_path", "query_text", 
        "model_response", "ground_truth_code", "code_generation_time", 
        "evaluation_result", "IoU_Score"
    ]
    df = df[[col for col in columns_order if col in df.columns]]
    
    out_name = csv_path.parent / f"{csv_path.stem}_GRADED.csv"
    df.to_csv(out_name, index=False)
    
    if len(is_correct_list) > 0:
        accuracy = (sum(is_correct_list) / len(is_correct_list)) * 100
        print(f"\n📊 Final Accuracy: {accuracy:.2f}%")
    print(f"💾 Graded results saved to: {out_name.name}")


# =====================================================================
# --- MAIN EXECUTION BLOCK ---
# =====================================================================

if __name__ == "__main__":
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    
    # Load manifest
    tb_df = load_turtlebench_json(TURTLEBENCH_PATH)
    
    if len(tb_df) > 0:
        print(f"✅ Loaded {len(tb_df)} tasks from JSON manifest.")
        
        for model in MODELS_TO_TEST:
            safe_name = model.replace("/", "_").replace(":", "_")
            csv_path = RESULTS_DIR / f"turtlebench_{safe_name}_generations.csv"
            
            # --- PHASE 1: GENERATE ---
            run_turtlebench_generation(tb_df, model, csv_path, limit=MAX_TASKS_TO_RUN)
            
            # --- PHASE 2: EVALUATE ---
            evaluate_generations(csv_path, TURTLEBENCH_PATH)
    else:
        print("❌ Dataset not loaded properly. Check TURTLEBENCH_PATH.")


✅ Loaded 260 tasks from JSON manifest.

🚀 TurtleBench Generation — gemma4:e2b...

Task 1 | Q: q1...
   ⏱️ Generated in 72.03s

🚀 Starting IoU execution on turtlebench_gemma4_e2b_generations.csv...
Evaluating Task 1...
   ⚠️ IoU Calculation Error: Unable to locate Ghostscript on paths
   ❌ IoU Match: 0.00%

📊 Final Accuracy: 0.00%
💾 Graded results saved to: turtlebench_gemma4_e2b_generations_GRADED.csv


# Setup & Configuration

In [1]:
# ─── Standard Library ───
import os
import re
import json
import time
import base64
from io import BytesIO
from pathlib import Path

# ─── Third-Party ───
import requests
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from PIL import Image
from openai import OpenAI

### Configurations

In [2]:
# ─── Local models to benchmark ───
MODELS_TO_TEST = ["gemma4:e2b"]

# ─── Retry budget for every API call ───
MAX_RETRIES = 3

# ─── Ollama-tunnelled client ───
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    timeout=180.0
)

# ─── Dataset Paths ───
CHARTQA_PATH    = Path("../DChartQA/chartqa.parquet")
AR_CHARTQA_PATH    = Path("../DChartQA/chartqa_arabic.parquet")
MATHVISION_PATH = Path("../DMathVision/mathvision.parquet")
AR_MATHVISION_PATH = Path("../DMathVision/mathvision_arabic.parquet")
SCREENQA_PATH   = Path("../DScreenQA/screenqa.parquet")
TURTLEBENCH_PATH = Path("../DTurtleBench")
PEARL_PATH      = Path("../DPEARL/data/pearl.parquet")

### Helper Functions

In [3]:
def encode_image_bytes_to_base64(byte_data, resize=None):
    """Convert raw image bytes to a base64 data-URL string.

    Parameters
    ----------
    byte_data : bytes
        Raw image bytes (e.g. from a parquet column).
    resize : tuple[int, int] | None
        Optional (width, height) cap; the image is thumbnailed to fit.

    Returns
    -------
    str | None
        A 'data:image/jpeg;base64,...' string, or None on failure.
    """
    try:
        img = Image.open(BytesIO(byte_data))
        if resize:
            img.thumbnail(resize)
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=85)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        return f"data:image/jpeg;base64,{b64}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None


def encode_raw_bytes_to_base64(byte_data):
    """Encode raw bytes directly to a base64 data-URL (no PIL processing)."""
    try:
        b64 = base64.b64encode(byte_data).decode("utf-8")
        return f"data:image/jpeg;base64,{b64}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None


def encode_image_file_to_base64(image_path, include_prefix=True):
    """Open an image file from disk, convert to JPEG base64.

    Parameters
    ----------
    image_path : str | Path
        Path to the image on disk.
    include_prefix : bool
        If True, return with 'data:image/jpeg;base64,' prefix.

    Returns
    -------
    str | None
    """
    try:
        with Image.open(image_path) as img:
            if img.mode != "RGB":
                img = img.convert("RGB")
            buf = BytesIO()
            img.save(buf, format="JPEG", quality=85)
            b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
            return f"data:image/jpeg;base64,{b64}" if include_prefix else b64
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None


def query_openrouter(prompt, base64_image, model_name=None, max_retries=MAX_RETRIES):
    """Send an image + prompt to OpenRouter and return (response_text, duration)."""
    model = model_name or MODEL_NAME
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": base64_image}},
            ],
        }],
        "temperature": 0.1,
    }

    start = time.time()
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers, json=payload, timeout=120,
            )
            resp.raise_for_status()
            text = resp.json()["choices"][0]["message"]["content"].strip()
            return text, time.time() - start
        except requests.exceptions.Timeout:
            print(f"   ⏳ Attempt {attempt + 1}/{max_retries}: timed out.")
        except requests.exceptions.RequestException as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: {e}")
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))

    return "ERROR_MAX_RETRIES_REACHED", time.time() - start


def query_openai_client(oai_client, model_name, prompt, base64_image, max_retries=MAX_RETRIES):
    """Send an image + prompt via an OpenAI-compatible client (Ollama / Ollama)."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": base64_image}},
        ],
    }]

    start = time.time()
    for attempt in range(max_retries):
        try:
            resp = oai_client.chat.completions.create(
                model=model_name, messages=messages,
                stream=False, temperature=0.1,
            )
            text = resp.choices[0].message.content.strip()
            return text, time.time() - start
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: {e}")
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))

    return "ERROR_MAX_RETRIES_REACHED", time.time() - start


def query_openai_client_text(oai_client, model_name, prompt, max_retries=MAX_RETRIES):
    """Send a text-only prompt via an OpenAI-compatible client."""
    start = time.time()
    for attempt in range(max_retries):
        try:
            resp = oai_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
            )
            return resp.choices[0].message.content.strip(), time.time() - start
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: {e}")
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
    return "ERROR_MAX_RETRIES_REACHED", time.time() - start


def load_progress(output_csv, key_columns):
    """Load already-processed task keys from an existing CSV to enable resuming.

    Parameters
    ----------
    output_csv : Path
        CSV file path.
    key_columns : list[str]
        Column(s) whose concatenation forms a unique task key.

    Returns
    -------
    tuple[set, bool]
        (processed_keys, file_exists)
    """
    processed = set()
    exists = output_csv.is_file()
    if exists:
        try:
            df = pd.read_csv(output_csv)
            if not df.empty and all(c in df.columns for c in key_columns):
                if len(key_columns) == 1:
                    processed = set(df[key_columns[0]].astype(str))
                else:
                    processed = set(
                        df[key_columns[0]].astype(str).str.cat(
                            [df[c].astype(str) for c in key_columns[1:]], sep="_"
                        )
                    )
                print(f"🔄 Resuming — skipping {len(processed)} already-processed tasks.")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV: {e}")
    return processed, exists


def append_result(result_dict, output_csv, file_exists):
    """Append a single result row to CSV, writing the header only once.

    Returns
    -------
    bool
        Updated file_exists flag (always True after first write).
    """
    pd.DataFrame([result_dict]).to_csv(
        output_csv, mode="a", header=not file_exists, index=False
    )
    return True


def safe_model_filename(name):
    """Sanitise a model name for use inside a filename."""
    return name.replace("/", "_").replace(":", "_")


def extract_image_bytes(row):
    """Extract raw image bytes from a parquet row (handles nested dict or flat column)."""
    if "decoded_image" in row and isinstance(row["decoded_image"], dict):
        return row["decoded_image"].get("bytes")
    if "image" in row and isinstance(row["image"], dict) and "bytes" in row["image"]:
        return row["image"]["bytes"]
    if "image" in row and isinstance(row["image"], bytes):
        return row["image"]
    return row.get("bytes")

# MathVision Evaluation

## Original

### Setup

In [ ]:
# ─── Load MathVision dataset ───
mathvision_df = None
if MATHVISION_PATH.exists():
    mathvision_df = pd.read_parquet(MATHVISION_PATH)
    print(f"✅ Loaded {len(mathvision_df)} rows.")
else:
    print(f"❌ Dataset not found at {MATHVISION_PATH}.")

### Prompt & Evaluator

In [ ]:
# ─── Execute ───
MATHVISION_OLLAMA_LIMIT = 2  # Set to None for full evaluation

MATHVISION_SYSTEM_PROMPT = """
Answer using only the image and text. 
RULES:
- Output EXACT final answer ONLY.
- NO explanations, reasoning, or filler words.
- Multiple choice: output the letter only (A, B, C, D).
- Math: output numbers or equations directly.
"""

def evaluate_math(ans, gt):
    """Check whether the model answer matches the ground truth (exact or substring)."""
    ans_clean = str(ans).lower().replace(" ", "")
    gt_clean = str(gt).lower().replace(" ", "")
    return ans_clean == gt_clean or gt_clean in ans_clean

In [ ]:
def run_mathvision_ollama(df, limit=None):
    """Run MathVision evaluation through Ollama-tunnelled backend."""
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 MathVision Ollama — {model_name}...")

        results_dir = Path("MathVisionResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"mathvision_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["id"])

        count = 0
        for _, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break
            task_id = str(row["id"])
            if task_id in processed:
                continue

            level   = str(row.get("level", "N/A"))
            subject = str(row.get("subject", "N/A"))
            query   = str(row["question"])
            gt      = str(row["answer"]).strip()

            print(f"\nTask {task_id} | {subject} | Lvl {level}")

            img_bytes = extract_image_bytes(row)
            b64 = encode_raw_bytes_to_base64(img_bytes) if img_bytes else None
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{query}\n{MATHVISION_SYSTEM_PROMPT}"
            answer, dur = query_openai_client(client, model_name, prompt, b64)
            passed = evaluate_math(answer, gt)

            file_exists = append_result({
                "id": task_id, "level": level, "subject": subject,
                "image": str(row.get("image", "N/A")), "question": query,
                "ground_truth": gt, "model_answer": answer,
                "evaluation_passed": passed, "run_time": round(dur, 3),
            }, output_csv, file_exists)

            processed.add(task_id)
            count += 1
            print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
            time.sleep(3)

if mathvision_df is not None:
    run_mathvision_ollama(mathvision_df, limit=MATHVISION_OLLAMA_LIMIT)
else:
    print("❌ mathvision_df not loaded — run the Setup cell first.")

## Arabic

### Setup

In [ ]:
# ─── Load MathVision dataset ───
mathvision_df = None
if AR_MATHVISION_PATH.exists():
    mathvision_df = pd.read_parquet(AR_MATHVISION_PATH)
    print(f"✅ Loaded {len(mathvision_df)} rows.")
else:
    print(f"❌ Dataset not found at {AR_MATHVISION_PATH}.")

### Prompt & Evaluator

In [ ]:
# ─── Execute ───
AR_MATHVISION_OLLAMA_LIMIT = 2  # Set to None for full evaluation

AR_MATHVISION_SYSTEM_PROMPT = """
أجب باستخدام الصورة والنص فقط.
قواعد:
- قدم الإجابة النهائية فقط.
- بدون شروحات، تبريرات، أو حشو.
- خيارات متعددة: اكتب الحرف فقط (A, B, C, D).
- الرياضيات: اكتب الأرقام أو المعادلات مباشرة.
"""


def evaluate_math(ans, gt):
    """Check whether the model answer matches the ground truth (exact or substring)."""
    ans_clean = str(ans).lower().replace(" ", "")
    gt_clean = str(gt).lower().replace(" ", "")
    return ans_clean == gt_clean or gt_clean in ans_clean

In [ ]:
def run_mathvision_ollama(df, limit=None):
    """Run Arabic MathVision evaluation through Ollama-tunnelled backend."""
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 Arabic MathVision Ollama — {model_name}...")

        results_dir = Path("MathVisionResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"mathvisionarabic_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["id"])

        count = 0
        for _, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break
            task_id = str(row["id"])
            if task_id in processed:
                continue

            level   = str(row.get("level", "N/A"))
            subject = str(row.get("subject", "N/A"))
            query   = str(row["arabic_question"])
            gt      = str(row["answer"]).strip()

            print(f"\nTask {task_id} | {subject} | Lvl {level}")

            img_bytes = extract_image_bytes(row)
            b64 = encode_raw_bytes_to_base64(img_bytes) if img_bytes else None
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{query}\n{AR_MATHVISION_SYSTEM_PROMPT}"
            answer, dur = query_openai_client(client, model_name, prompt, b64)
            passed = evaluate_math(answer, gt)

            file_exists = append_result({
                "id": task_id, "level": level, "subject": subject,
                "image": str(row.get("image", "N/A")), "question": query,
                "ground_truth": gt, "model_answer": answer,
                "evaluation_passed": passed, "run_time": round(dur, 3),
            }, output_csv, file_exists)

            processed.add(task_id)
            count += 1
            print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
            time.sleep(3)

if mathvision_df is not None:
    run_mathvision_ollama(mathvision_df, limit=AR_MATHVISION_OLLAMA_LIMIT)
else:
    print("❌ mathvision_df not loaded — run the Setup cell first.")

# ChartQA Evaluation

In [4]:
def normalize_arabic_text(text):
    """Converts Eastern Arabic numerals to Western, and normalizes Arabic punctuation."""
    arabic_to_western = {
        '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', 
        '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9',
        '٪': '%', '،': ',', '٫': '.' 
    }
    trans_table = str.maketrans(arabic_to_western)
    return str(text).translate(trans_table)

def evaluate_chartqa(ans, gt):
    """Robust ChartQA evaluation handling English, Arabic, visual tolerance, and the Year trap."""
    
    ans_norm = normalize_arabic_text(ans)
    ans_clean = str(ans_norm).lower().strip()
    
    valid_labels = gt if isinstance(gt, list) else [str(gt)]
    
    for label in valid_labels:
        label_norm = normalize_arabic_text(label)
        label_clean = str(label_norm).lower().strip()
        
        # 1. Exact String Match
        if ans_clean == label_clean:
            return True
            
        # 2. Numeric Checks
        ans_num_str = re.sub(r'[\$\£\€\,\%]', '', ans_clean)
        gt_num_str = re.sub(r'[\$\£\€\,\%]', '', label_clean)
        
        try:
            ans_float = float(ans_num_str)
            gt_float = float(gt_num_str)
            
            # --- THE FIX: The Year Trap ---
            # If the ground truth is an integer that looks like a year, force exact match.
            if gt_float.is_integer() and 1800 <= gt_float <= 2100:
                if ans_float == gt_float:
                    return True
                # If it doesn't match exactly, we skip the 5% block below
            else:
                # 5% visual estimation tolerance check for normal data values
                if gt_float == 0:
                    if ans_float == 0: 
                        return True
                elif abs(ans_float - gt_float) / abs(gt_float) <= 0.05:
                    return True
        except ValueError:
            pass 

        # 3. Safe Substring Match 
        if label_clean.isdigit():
            if re.search(fr'\b{label_clean}\b', ans_clean):
                return True
        else:
            if label_clean in ans_clean:
                return True
                
    return False


## Original

In [ ]:
# ─── Load ChartQA dataset ───
chartqa_df = None
if CHARTQA_PATH.exists():
    chartqa_df = pd.read_parquet(CHARTQA_PATH)
    print(f"✅ Loaded {len(chartqa_df)} rows.")
    print(f"Columns: {list(chartqa_df.columns)}")
else:
    print(f"❌ Dataset not found at {CHARTQA_PATH}.")

In [ ]:
# ─── ChartQA run variables ───
CHARTQA_OLLAMA_LIMIT = 2 # Set to None for full evaluation

CHARTQA_SYSTEM_PROMPT = """
Answer using ONLY chart data.
RULES:
- Output ONLY the raw final answer. Zero filler or reasoning.
- Numbers: Use digits + units/currencies if applicable (e.g., 45%, $120). Do not spell out.
- Booleans: Output strictly "Yes" or "No".
- Lists: Comma-separated (e.g., A, B, C).
- Labels: Exact text match from the image.

"""

### Ollama

In [ ]:
def run_chartqa_ollama(df, limit=None):
    """Run ChartQA evaluation through Ollama-tunnelled backend."""
    for model_name in MODELS_TO_TEST:
        print(f"🚀 ChartQA Ollama — {model_name}...")

        results_dir = Path("ChartQAResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"chartqa_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["imgname", "query"])

        count = 0
        for index, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break

            imgname  = str(row["imgname"])
            query    = str(row["query"])
            label    = row["label"]
            qa_type  = str(row["type"])
            task_key = f"{imgname}_{query}"

            if task_key in processed:
                continue

            print(f"[{index}] Image: {imgname} | Query: {query[:60]}")

            b64 = encode_image_bytes_to_base64(row["image"])
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{query}\n{CHARTQA_SYSTEM_PROMPT}"
            answer, dur = query_openai_client(client, model_name, prompt, b64)
            passed = evaluate_chartqa(answer, label)

            file_exists = append_result({
                "id": str(index), "imgname": imgname, "query": query,
                "ground_truth": str(label), "type": qa_type,
                "model_answer": answer, "evaluation_passed": passed,
                "run_time": round(dur, 3),
            }, output_csv, file_exists)

            processed.add(task_key)
            count += 1
            print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}' | GT: '{label}'")
            time.sleep(3)

# ─── Execute ───
if chartqa_df is not None:
    run_chartqa_ollama(chartqa_df, limit=CHARTQA_OLLAMA_LIMIT)
else:
    print("❌ chartqa_df not loaded — run the Setup cell first.")

## Arabic

In [5]:
# ─── Load ChartQA dataset ───
chartqa_df = None
if AR_CHARTQA_PATH.exists():
    chartqa_df = pd.read_parquet(AR_CHARTQA_PATH)
    chartqa_df['arabic_ground_truth'] = chartqa_df['arabic_ground_truth'].astype(str)
    print(f"✅ Loaded {len(chartqa_df)} rows.")
    print(f"Columns: {list(chartqa_df.columns)}")
else:
    print(f"❌ Dataset not found at {AR_CHARTQA_PATH}.")

✅ Loaded 32719 rows.
Columns: ['imgname', 'query', 'label', 'type', 'image', 'arabic_query', 'arabic_ground_truth']


In [7]:
# ─── ChartQA run variables ───
AR_CHARTQA_OLLAMA_LIMIT = 4 # Set to None for full evaluation

AR_CHARTQA_SYSTEM_PROMPT = """
أجب باستخدام بيانات الرسم البياني فقط.
قواعد:
- اكتب الإجابة النهائية المجردة فقط. بدون حشو أو تفسير.
- الأرقام: استخدم الأرقام + الوحدات/العملات إن وجدت (مثل 45%، $120). لا تكتبها حروفاً.
- نعم/لا: اكتب "نعم" أو "لا" فقط.
- القوائم: افصل بينها بفاصلة (مثل أ، ب، ج).
- التسميات: نسخ دقيق للنص كما يظهر في الصورة.

"""

### Ollama

In [ ]:
def run_chartqa_ollama(df, limit=None):
    """Run ChartQA evaluation through Ollama-tunnelled backend."""
    for model_name in MODELS_TO_TEST:
        print(f"🚀 ChartQA Ollama — {model_name}...")

        results_dir = Path("ChartQAResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"chartqaarabic_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["imgname", "query"])

        count = 0
        for index, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break

            imgname  = str(row["imgname"])
            query    = str(row["arabic_query"])
            label    = row["arabic_ground_truth"]
            qa_type  = str(row["type"])
            task_key = f"{imgname}_{query}"

            if task_key in processed:
                continue

            print(f"[{index}] Image: {imgname} | Query: {query[:60]}")

            b64 = encode_image_bytes_to_base64(row["image"])
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{query}\n{AR_CHARTQA_SYSTEM_PROMPT}"
            answer, dur = query_openai_client(client, model_name, prompt, b64)
            passed = evaluate_chartqa(answer, label)

            file_exists = append_result({
                "id": str(index), "imgname": imgname, "query": query,
                "ground_truth": str(label), "type": qa_type,
                "model_answer": answer, "evaluation_passed": passed,
                "run_time": round(dur, 3),
            }, output_csv, file_exists)

            processed.add(task_key)
            count += 1
            print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}' | GT: '{label}'")
            time.sleep(3)

# ─── Execute ───
if chartqa_df is not None:
    run_chartqa_ollama(chartqa_df, limit=AR_CHARTQA_OLLAMA_LIMIT)
else:
    print("❌ chartqa_df not loaded — run the Setup cell first.")

# TurtleBench Evaluation

### Prompt & Evaluator

In [ ]:
TURTLEBENCH_SYSTEM_PROMPT = """
INSTRUCTIONS:
1. Write the Python Turtle graphics code to generate the requested image.
2. Output ONLY the raw Python code with no setup for execution.
3. DO NOT wrap the code in markdown formatting (e.g., do not use ```python).
4. DO NOT output any conversational text, explanations, or greetings.
5. Assume the turtle object is named `t` and the math module is imported if needed.
"""

TURTLEBENCH_EVAL_PROMPT_TEMPLATE = """Compare the following two Python Turtle scripts.
Analyze their logic and determine if they will generate the exact same visual shape/output on the screen.
Answer with 'Yes' or 'No' followed by one statment explaining the difference. Do not exceed one statment and do not give a long statment.

Model Generated Code:
{model_code}

Ground Truth Code:
{ground_truth_code}"""


def evaluate_turtle_equivalence_openrouter(model_code, gt_code):
    """Use OpenRouter to judge whether two Turtle scripts produce the same output."""
    prompt = TURTLEBENCH_EVAL_PROMPT_TEMPLATE.format(
        model_code=model_code, ground_truth_code=gt_code
    )
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
    }
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers, json=payload, timeout=120,
            )
            resp.raise_for_status()
            raw = resp.json()["choices"][0]["message"]["content"].strip().lower()
            if "yes" in raw:
                return True
            if "no" in raw:
                return False
            return raw
        except Exception as e:
            print(f"   ❌ Eval attempt {attempt + 1}/{MAX_RETRIES}: {e}")
        if attempt < MAX_RETRIES - 1:
            time.sleep(5 * (attempt + 1))
    return "ERROR"


def evaluate_turtle_equivalence_client(oai_client, model_name, model_code, gt_code):
    """Use an OpenAI-compatible client to judge Turtle script equivalence."""
    prompt = TURTLEBENCH_EVAL_PROMPT_TEMPLATE.format(
        model_code=model_code, ground_truth_code=gt_code
    )
    for attempt in range(MAX_RETRIES):
        try:
            resp = oai_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
            )
            raw = resp.choices[0].message.content.strip().lower()
            if "yes" in raw:
                return True
            if "no" in raw:
                return False
            return raw
        except Exception as e:
            print(f"   ❌ Eval attempt {attempt + 1}/{MAX_RETRIES}: {e}")
        if attempt < MAX_RETRIES - 1:
            time.sleep(5 * (attempt + 1))
    return "ERROR" 

### Dataset Loader

In [ ]:
def load_turtle_bench(dataset_path):
    """Walk the DTurtleBench/Tasks folder tree and return a DataFrame of questions."""
    tasks_dir = Path(dataset_path) / "Tasks"
    if not tasks_dir.exists():
        print(f"❌ Tasks folder not found at {tasks_dir}")
        return None

    rows = []
    for task_folder in sorted(tasks_dir.iterdir()):
        if not task_folder.is_dir():
            continue
        task_id    = task_folder.name
        text_dir   = task_folder / "QA" / "text"
        code_dir   = task_folder / "QA" / "code"
        image_path = task_folder / "image" / f"{task_id}.png"

        if not text_dir.exists():
            continue
        for qfile in sorted(text_dir.glob("q*.txt")):
            qid = qfile.stem
            query_text = qfile.read_text(encoding="utf-8").strip()
            code_file  = code_dir / f"{qid}_code.txt"
            code_text  = code_file.read_text(encoding="utf-8").strip() if code_file.exists() else "No code found."

            rows.append({
                "task_id": task_id, "question_id": qid,
                "query_text": query_text, "code_text": code_text,
                "task_folder": str(task_folder), "query_path": str(qfile),
                "code_path": str(code_file), "image_path": str(image_path),
            })
    return pd.DataFrame(rows)


def find_turtlebench_path():
    """Auto-detect the DTurtleBench folder by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if parent.name.lower() == "thesis":
            candidate = parent / "DTurtleBench"
            if candidate.exists():
                return candidate
    # Absolute fallback
    fallback = Path("/Users/smeh/Desktop/Thesis/DTurtleBench")
    return fallback if fallback.exists() else None

### Ollama

In [ ]:
# ─── TurtleBench run variables (Ollama) ───
TURTLEBENCH_OLLAMA_LIMIT = 2  # Set to None for full evaluation

def run_turtlebench_ollama(df, limit=None):
    """Run TurtleBench evaluation through Ollama-tunnelled backend."""
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 TurtleBench Ollama — {model_name}...")

        results_dir = Path("TurtleBenchResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"turtlebench_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["task_id", "question_id"])

        eval_df = df.head(limit) if limit else df
        for _, row in eval_df.iterrows():
            task_key = f"{row['task_id']}_{row['question_id']}"
            if task_key in processed:
                continue

            print(f"\nTask {row['task_id']} | Q: {row['question_id']}...")

            # Ollama expects raw base64 without prefix
            b64 = encode_image_file_to_base64(row["image_path"], include_prefix=False)
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{row['query_text']}\n{TURTLEBENCH_SYSTEM_PROMPT}"
            b64_url = f"data:image/jpeg;base64,{b64}"
            model_code, gen_time = query_openai_client(client, model_name, prompt, b64_url)
            print(f"   ⏱️ Generated in {gen_time:.2f}s")
            time.sleep(2)

            gt_code = row["code_text"].strip()
            print("   🔍 Evaluating code equivalence...")
            passed = evaluate_turtle_equivalence_client(client, model_name, model_code, gt_code)

            file_exists = append_result({
                "task_id": row["task_id"], "question_id": row["question_id"],
                "image_path": row["image_path"], "query_text": row["query_text"],
                "model_response": model_code, "ground_truth_code": gt_code,
                "code_generation_time": gen_time, "evaluation_passed": passed,
            }, output_csv, file_exists)

            processed.add(task_key)
            print(f"   {'✅' if passed is True else '❌'} Done.")
            time.sleep(2)


# ─── Execute ───
_tb_path = find_turtlebench_path()
if _tb_path:
    _tb_df = load_turtle_bench(_tb_path)
    if _tb_df is not None and not _tb_df.empty:
        print(f"✅ Loaded {len(_tb_df)} questions.")
        run_turtlebench_ollama(_tb_df, limit=TURTLEBENCH_OLLAMA_LIMIT)
    else:
        print("⚠️ No TurtleBench questions found.")
else:
    print("❌ DTurtleBench folder not found.")

# ScreenQA Evaluation

### Setup

In [ ]:
# ─── Load ScreenQA dataset ───
screenqa_df = None
if SCREENQA_PATH.exists():
    metadata = pq.read_metadata(SCREENQA_PATH)
    print(f"📊 ScreenQA dataset has {metadata.num_rows} rows.")
    screenqa_df = pq.read_table(SCREENQA_PATH).to_pandas()
else:
    print(f"❌ Dataset not found at {SCREENQA_PATH}.")

### Prompt & Evaluator

In [ ]:
SCREENQA_SYSTEM_PROMPT = """
You are an expert visual analyst. Answer the user's question using only the provided image and text.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the image").
"""

# ─── Image resize dimensions for ScreenQA ───
SCREENQA_RESIZE = (1280, 720)


def evaluate_screenqa(ans, gt):
    """Relaxed match with currency/symbol stripping for screen-based QA."""
    ans_str = re.sub(r'[\$\£\€\,]', '', str(ans).lower())
    gt_str  = re.sub(r'[\$\£\€\,]', '', str(gt).lower())

    ans_clean = re.sub(r'[^\w]', '', ans_str)
    gt_clean  = re.sub(r'[^\w]', '', gt_str)

    if not ans_clean or not gt_clean:
        return False
    if ans_clean == gt_clean:
        return True
    if gt_clean in ans_clean:
        return True
    # Prevent single-char false positives (e.g. "1" matching "100")
    if len(ans_clean) >= 2 and ans_clean in gt_clean:
        return True
    return False


def extract_screenqa_ground_truth(row):
    """Safely extract the ground-truth answer from ScreenQA's nested structure."""
    raw = row.get("answers", row.get("ground_truth", row.get("full_answer", [])))
    try:
        return str(raw[0]["full_answer"]).strip()
    except (IndexError, KeyError, TypeError):
        return str(raw).strip()

### Ollama

In [ ]:
# ─── ScreenQA run variables (Ollama) ───
SCREENQA_OLLAMA_LIMIT = 2  # Set to None for full evaluation

In [ ]:
def run_screenqa_ollama(df, limit=None):
    """Run ScreenQA evaluation through Ollama-tunnelled backend."""
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 ScreenQA Ollama — {model_name}...")

        results_dir = Path("ScreenQAResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"screenqa_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["id"])

        count = 0
        for _, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break

            task_id = str(row["screen_id"])
            if task_id in processed:
                continue

            image_name = str(row.get("file_name", "N/A"))
            query = str(row["question"])
            gt = extract_screenqa_ground_truth(row)

            print(f"\nTask {task_id} | Image: {image_name}")

            img_bytes = extract_image_bytes(row)
            b64 = encode_image_bytes_to_base64(img_bytes, resize=SCREENQA_RESIZE) if img_bytes else None
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{query}\n{SCREENQA_SYSTEM_PROMPT}"
            answer, dur = query_openai_client(client, model_name, prompt, b64)
            passed = evaluate_screenqa(answer, gt)

            file_exists = append_result({
                "id": task_id, "image": image_name, "question": query,
                "ground_truth": gt, "model_answer": answer,
                "evaluation_passed": passed, "run_time": round(dur, 3),
            }, output_csv, file_exists)

            processed.add(task_id)
            count += 1
            print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
            time.sleep(3)


# ─── Execute ───
if screenqa_df is not None:
    run_screenqa_ollama(screenqa_df, limit=SCREENQA_OLLAMA_LIMIT)
else:
    print("❌ screenqa_df not loaded — run the Setup cell first.")

# PEARL Evaluation

### Setup

In [ ]:
# ─── Load PEARL dataset ───
pearl_df = None
if PEARL_PATH.exists():
    pearl_df = pd.read_parquet(PEARL_PATH)
    print(f"✅ Loaded {len(pearl_df)} rows.")
else:
    print(f"❌ Dataset not found at {PEARL_PATH}.")

### Prompt & Evaluator

In [ ]:
PEARL_SYSTEM_PROMPT = (
    "You are an expert visual analyst. Answer the user's question using only the provided image and text.\n\n"
    "CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:\n"
    "1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler "
    "(e.g., never write \"The answer is\" or \"Based on the image\").\n"
    "2. IMPORTANT: You MUST answer in the EXACT SAME LANGUAGE as the question. "
    "If the question is in Arabic, answer in Arabic. If in English, answer in English, etc.\n"
    "If it is a true and false question answer TRUE or FALSE, if it a multiple choice question give me the answer itself and NOT the letter.\n"
)


def evaluate_pearl(ans, gt):
    """Exact or substring match for PEARL answers."""
    a = str(ans).strip()
    g = str(gt).strip()
    if not a or not g:
        return False
    if a == g:
        return True
    if g in a:
        return True
    if len(a) >= 2 and a in g:
        return True
    return False

### Ollama

In [ ]:
# ─── PEARL run variables (Ollama) ───
PEARL_OLLAMA_LIMIT = 2  # Set to None for full evaluation

In [ ]:
def run_pearl_ollama(df, limit=None):
    """Run PEARL evaluation through local Ollama."""
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 PEARL Ollama — {model_name}...")

        results_dir = Path("PEARLResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"pearl_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["index"])

        count = 0
        for idx, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break

            idx_str = str(idx)
            if idx_str in processed:
                continue

            query    = str(row["question"]).strip()
            gt       = str(row["answer"]).strip()
            category = str(row.get("category", "N/A"))
            image_id = str(row.get("image_id", "N/A"))

            print(f"\nTask {idx_str} | image_id: {image_id} | category: {category}")

            img_bytes = extract_image_bytes(row)
            b64 = encode_image_bytes_to_base64(img_bytes) if img_bytes else None
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{query}\n\n{PEARL_SYSTEM_PROMPT}"
            answer, dur = query_openai_client(client, model_name, prompt, b64)
            passed = evaluate_pearl(answer, gt)

            file_exists = append_result({
                "index": idx_str, "image_id": image_id, "category": category,
                "question": query, "ground_truth": gt,
                "model_answer": answer, "evaluation_passed": passed,
                "run_time": round(dur, 3),
            }, output_csv, file_exists)

            processed.add(idx_str)
            count += 1
            print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
            time.sleep(3)


# ─── Execute ───
if pearl_df is not None:
    run_pearl_ollama(pearl_df, limit=PEARL_OLLAMA_LIMIT)
else:
    print("❌ pearl_df not loaded — run the Setup cell first.")

# Results Evaluation

In [ ]:
# ─── Folders to scan for result CSVs ───
RESULTS_FOLDERS = [
    "MathVisionResults", "ChartQAResults", "ScreenQAResults",
    "TurtleBenchResults", "MTVQAResults", "PEARLResults",
]

In [ ]:
def summarize_results(folders=RESULTS_FOLDERS):
    """Print a table of accuracy percentages for every result CSV found."""
    print(f"{'Model Name':<60} | {'Accuracy':>10}")
    print("-" * 75)

    for folder in folders:
        folder_path = Path(folder)
        if not folder_path.exists():
            continue
        for csv_file in sorted(folder_path.glob("*.csv")):
            try:
                df = pd.read_csv(csv_file)
                if "evaluation_passed" not in df.columns or len(df) == 0:
                    continue
                correct  = df["evaluation_passed"].astype(int).sum()
                accuracy = (correct / len(df)) * 100
                name = csv_file.stem.replace("_ollama", "").replace("_results", "")
                print(f"{name:<60} | {accuracy:>8.2f}%")
            except Exception as e:
                print(f"⚠️ Error processing {csv_file.name}: {e}")


summarize_results()